<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/05_Hands_On_Labs/notebooks/02_stats_intuition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Statistical Intuition: Distributions, CLT & Hypothesis Testing

> **Track:** Data Science | **Level:** Beginner | **Time:** 2 hours

## Overview
Statistics is the language of data science. This notebook builds **intuition** for the concepts that appear in almost every data project: probability distributions, the Central Limit Theorem, and hypothesis testing.

### What You'll Learn
- Normal, binomial, and Poisson distributions
- The Central Limit Theorem (with visual proof)
- z-scores and standardization
- p-values and significance
- t-tests and chi-square tests
- Common statistical mistakes

```bash
pip install numpy scipy matplotlib seaborn
```

In [ ]:
# Setup: imports and random seed for reproducibility
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)

print("Libraries loaded. Let's build statistical intuition!")

## 1. Probability Distributions

Three distributions appear constantly in data science:
- **Normal (Gaussian)**: heights, measurement errors, many natural phenomena
- **Binomial**: number of successes in N trials (clicks, conversions)
- **Poisson**: number of events in a fixed interval (requests/second, defects/hour)

In [ ]:
# Visualize the three core distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Normal distribution
x_norm = np.linspace(-4, 4, 300)
axes[0].plot(x_norm, stats.norm.pdf(x_norm, 0, 1), 'b-', linewidth=2)
axes[0].fill_between(x_norm, stats.norm.pdf(x_norm, 0, 1), alpha=0.3)
axes[0].set_title("Normal Distribution (μ=0, σ=1)")
axes[0].set_xlabel("Value")
axes[0].axvline(0, color='red', linestyle='--', label='mean')
axes[0].legend()

# Binomial distribution (n=20, p=0.3 — like a 30% conversion rate)
x_binom = np.arange(0, 21)
axes[1].bar(x_binom, stats.binom.pmf(x_binom, n=20, p=0.3), color='green', alpha=0.7)
axes[1].set_title("Binomial (n=20, p=0.3): Conversions in 20 visitors")
axes[1].set_xlabel("Number of conversions")

# Poisson distribution (λ=5 — like 5 requests/second average)
x_pois = np.arange(0, 16)
axes[2].bar(x_pois, stats.poisson.pmf(x_pois, mu=5), color='orange', alpha=0.7)
axes[2].set_title("Poisson (λ=5): Requests per second")
axes[2].set_xlabel("Number of events")

plt.tight_layout()
plt.show()

print(f"Normal: P(-1 < X < 1) = {stats.norm.cdf(1) - stats.norm.cdf(-1):.1%} (the 68% rule)")
print(f"Binomial mean: n*p = {20*0.3}, Poisson mean = λ = 5")

## 2. The Central Limit Theorem (Visual Proof)

**The CLT**: No matter what distribution a population has, the **sampling distribution of the mean** approaches normal as sample size increases. This is why statistics works on real-world messy data.

In [ ]:
# Demonstrate the Central Limit Theorem visually
# Start with a very non-normal distribution (exponential)
population = np.random.exponential(scale=2, size=100_000)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
sample_sizes = [1, 5, 30, 100]

for ax, n in zip(axes, sample_sizes):
    # Take 1000 samples of size n and compute their means
    sample_means = [np.mean(np.random.choice(population, size=n)) for _ in range(1000)]
    ax.hist(sample_means, bins=40, density=True, alpha=0.7, color='steelblue')
    ax.set_title(f"Sample size n={n}")
    ax.set_xlabel("Sample mean")

axes[0].set_ylabel("Density")
fig.suptitle("CLT Demo: Population is Exponential, but Sample Means → Normal", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Population shape: right-skewed exponential (mean={population.mean():.2f}, std={population.std():.2f})")
print("By n=30, sample means are approximately normal — the CLT at work!")

## 3. Hypothesis Testing

Hypothesis testing answers: "Is this difference **real** or just random chance?"
- **Null hypothesis (H₀)**: no effect, no difference
- **Alternative hypothesis (H₁)**: there is a real effect
- **p-value**: probability of seeing this result (or more extreme) if H₀ is true

In [ ]:
# Practical hypothesis testing: t-test and chi-square

# Example 1: Did a product change improve user session duration?
control_sessions = np.random.normal(loc=5.2, scale=1.5, size=200)   # minutes
treatment_sessions = np.random.normal(loc=5.8, scale=1.5, size=200)  # slightly higher

t_stat, p_value = stats.ttest_ind(control_sessions, treatment_sessions)
print("=== Independent t-test: Session Duration ===")
print(f"Control mean: {control_sessions.mean():.2f} min | Treatment mean: {treatment_sessions.mean():.2f} min")
print(f"t-statistic: {t_stat:.3f} | p-value: {p_value:.4f}")
print(f"Result: {'Significant difference (p<0.05)' if p_value < 0.05 else 'No significant difference'}")

print()

# Example 2: Chi-square test — is click rate independent of button color?
# Observed: [clicks, no-clicks] for [red, blue, green] buttons
observed = np.array([[120, 380], [160, 340], [100, 400]])
chi2, p_chi, dof, expected = stats.chi2_contingency(observed)
print("=== Chi-Square Test: Button Color vs Click Rate ===")
print(f"Chi² statistic: {chi2:.3f} | p-value: {p_chi:.4f} | df: {dof}")
print(f"Result: {'Button color affects click rate!' if p_chi < 0.05 else 'No significant effect of color'")

## 4. Common Statistical Mistakes

Understanding these mistakes will save you from drawing wrong conclusions in real projects:

In [ ]:
# Demonstrate p-hacking and the multiple comparisons problem

# Mistake 1: p-hacking — run enough tests and you'll find "significance" by chance
np.random.seed(0)
false_positives = 0
n_experiments = 100
for _ in range(n_experiments):
    # Two identical groups — no real difference!
    group_a = np.random.normal(0, 1, 30)
    group_b = np.random.normal(0, 1, 30)
    _, p = stats.ttest_ind(group_a, group_b)
    if p < 0.05:
        false_positives += 1

print("=== Common Statistical Mistakes ===")
print(f"\n1. P-HACKING / Multiple Comparisons:")
print(f"   Ran {n_experiments} tests on IDENTICAL populations.")
print(f"   'Significant' results (p<0.05): {false_positives}/{n_experiments} = {false_positives}%")
print(f"   Expected by chance: ~5%. Bonferroni correction: use α={0.05/n_experiments:.4f}")

# Mistake 2: Correlation ≠ Causation
print(f"\n2. CORRELATION ≠ CAUSATION:")
x = np.arange(10)
ice_cream = x * 3 + np.random.normal(0, 1, 10)
drownings = x * 2 + np.random.normal(0, 1, 10)
r, p = stats.pearsonr(ice_cream, drownings)
print(f"   Ice cream sales vs drownings: r={r:.2f}, p={p:.4f}")
print(f"   Both are driven by summer heat! (confounding variable)")

# Mistake 3: Small samples
print(f"\n3. SMALL SAMPLES MISLEAD:")
for n in [5, 30, 200]:
    sample = np.random.normal(100, 15, n)  # IQ scores, true mean=100
    ci = stats.t.interval(0.95, df=n-1, loc=np.mean(sample), scale=stats.sem(sample))
    print(f"   n={n:4d}: mean={np.mean(sample):.1f}, 95% CI: [{ci[0]:.1f}, {ci[1]:.1f}] (width={ci[1]-ci[0]:.1f})")

## Exercises

1. **Power analysis**: Implement a function `minimum_sample_size(effect_size, alpha=0.05, power=0.8)` using `scipy.stats` that calculates the minimum N needed to detect a given effect. How many users do you need to detect a 10% lift in conversion rate (from 5% to 5.5%)?

2. **Bootstrap confidence intervals**: Without using `scipy.stats.t.interval`, implement bootstrap resampling to compute a 95% confidence interval for the mean of a dataset. Compare it to the parametric CI for samples of size n=10 and n=100.

3. **Bayesian vs Frequentist**: Implement a simple **Bayesian A/B test** using Beta distributions. Start with a Beta(1,1) prior and update it with conversion data (successes + failures). Plot the posterior distributions for two variants and compute the probability that variant B is better than A.